## Quick Exploratory Data Analysis

### 1. Importar

In [ ]:
import pandas as pd
import json

### 2. Load

In [ ]:
categories = pd.read_csv("../data/desafio_tecnico_ai_eng - categories.csv")
customers = pd.read_csv("../data/desafio_tecnico_ai_eng - customers.csv")
products = pd.read_csv("../data/desafio_tecnico_ai_eng - products.csv")
orders = pd.read_csv("../data/desafio_tecnico_ai_eng - orders.csv")
order_items = pd.read_csv("../data/desafio_tecnico_ai_eng - order_items.csv")
promotions = pd.read_csv("../data/desafio_tecnico_ai_eng - promotions.csv")

promotions.head(2)

In [ ]:
print("SHAPE DOS DATASETS")
for name, df in [("categories", categories), ("customers", customers),
                 ("products", products), ("orders", orders),
                 ("order_items", order_items), ("promotions", promotions)]:
    print(f"{name}: {df.shape[0]} linhas x {df.shape[1]} colunas")

## 3. Produtos por Categoria

In [ ]:
merged = products.merge(categories, on="category_id", suffixes=("", "_cat"))
cat_summary = merged.groupby("name_cat").agg(
    qtd_produtos=("product_id", "count"),
    preco_min=("price_brl", "min"),
    preco_max=("price_brl", "max"),
    preco_medio=("price_brl", lambda x: round(x.mean(), 2)),
    estoque_total=("stock_quantity", "sum")
).sort_values("qtd_produtos", ascending=False)

print("\n=== PRODUTOS POR CATEGORIA ===")
print(cat_summary)

print("\n Categorias SEM produtos:",
      list(set(categories["name"]) - set(merged["name_cat"])))

## 4. Status dos Produtos

In [ ]:
products.head(2)

In [ ]:
print(products["status"].value_counts())

## 4.1 Pontos de Atenção - Estoque zero mas status active

In [ ]:
zero_stock_active = products[(products["stock_quantity"] == 0) & (products["status"] == "active")]
print(f"\n4.1 - Estoque ZERO com status 'active' ({len(zero_stock_active)} produtos):")
print(zero_stock_active[["product_id", "name", "stock_quantity", "status"]].to_string(index=False))
print("Tratar como 'temporariamente indisponível' na tool")

## 4.2 Pontos de Atenção - Produtos descontinuados

In [ ]:
discontinued = products[products["status"] == "discontinued"]
print(f"\n4.2 - Produtos DESCONTINUADOS ({len(discontinued)}):")
print(discontinued[["product_id", "name", "status"]].to_string(index=False))
print("Não mostrar como opção de compra, sugerir alternativa")

## 4.3 Pontos de Atenção - Produtos coming_soon

In [ ]:
coming = products[products["status"] == "coming_soon"]
print(f"\n4.3 - Produtos COMING SOON ({len(coming)}):")
print(coming[["product_id", "name", "status"]].to_string(index=False))
print("Informar que está chegando, sem permitir compra")

## 4.4 Pontos de Atenção - Promoções ativas vs expiradas

In [ ]:
active_promos = promotions[promotions["is_active"] == 1]
inactive_promos = promotions[promotions["is_active"] == 0]

print(f"\n4.4 - Promoções: {len(active_promos)} ativas / {len(inactive_promos)} expiradas")
print("Promoções ATIVAS:")
active_detail = active_promos.merge(products[["product_id", "name", "price_brl"]], on="product_id")
active_detail["preco_com_desconto"] = round(active_detail["price_brl"] * (1 - active_detail["discount_percent"]/100), 2)
print(active_detail[["promotion_id", "name", "price_brl", "discount_percent", "preco_com_desconto", "description"]].to_string(index=False))
print("Promoções expiradas: informar que não está mais vigente e dar preço atual")

## 4.5 Pontos de Atenção - Inconsistência pedido x produto coming_soon

In [ ]:
print("\n4.5 - Inconsistência: pedido com produto coming_soon")
order3_items = order_items[order_items["order_id"] == 3]
order3_products = order3_items.merge(products[["product_id", "name", "status"]], on="product_id")
print(order3_products[["order_id", "product_id", "quantity", "name", "status"]].to_string(index=False))
print("Dados com inconsistência pontual, manter como fornecido")

In [ ]:
# ressalto que no ponto acima o pedido pode ter sido feito e o cliente está ciente que ele está 'coming soon'

## 5. Pedidos 

In [ ]:
print(f"Total: {len(orders)}")
print(orders["status"].value_counts())
print(f"\nMétodos de pagamento:")
print(orders["payment_method"].value_counts())

## 6. Clientes

In [ ]:
print("\n=== CLIENTES ===")
print(f"Total: {len(customers)}")
print(f"Cidades:")
print(customers["city"].value_counts())